# SageMaker Demo: Employee Attrition Prediction Using Feature Store and XGBoost

This notebook demonstrates how to use Amazon SageMaker's Feature Store and XGBoost built-in algorithm to predict employee attrition.

In [ ]:
!pip uninstall -y sagemaker

In [ ]:
!rm -rf /opt/conda/lib/python3.12/site-packages/sagemaker
!rm -rf /opt/conda/lib/python3.12/site-packages/sagemaker-*.dist-info

In [ ]:
!pip install sagemaker==2.224.0 --no-cache-dir

In [ ]:
import sagemaker

print(sagemaker.__file__)
print(dir(sagemaker)[:30])

In [ ]:
import sagemaker

print(sagemaker.__version__)
print(hasattr(sagemaker, "Session"))
print(hasattr(sagemaker, "get_execution_role"))

In [1]:
# Step 1: Setup
import sagemaker
import boto3
import pandas as pd

# Start SageMaker session
sagemaker_session = sagemaker.Session()

# Get execution role
role = sagemaker.get_execution_role()

# Get AWS region
region = boto3.Session().region_name

# S3 bucket for storing data
bucket = 'training-data-ar' #change the bucket name here
# Load dataset
file_path = 'Employee.csv'

employee_df = pd.read_csv(file_path)

# Display first rows
employee_df.head()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


In [2]:
# Step 2: Data Preparation
# Convert categorical columns to numeric
employee_df['Education'] = employee_df['Education'].astype('category').cat.codes
employee_df['City'] = employee_df['City'].astype('category').cat.codes
employee_df['Gender'] = employee_df['Gender'].astype('category').cat.codes
employee_df['EverBenched'] = employee_df['EverBenched'].map({'Yes': 1, 'No': 0})

# Drop rows with NaN values in the target column
employee_df.dropna(subset=['LeaveOrNot'])

# Convert target column to numeric if needed
employee_df['LeaveOrNot'] = employee_df['LeaveOrNot'].astype(int)

# Ensure no missing values in feature columns
employee_df = employee_df.dropna()

# Verify all columns are numeric
print(employee_df.dtypes)

# Define features and target
feature_columns = [
    'Education', 'JoiningYear', 'City', 'PaymentTier', 'Age',
    'Gender', 'EverBenched', 'ExperienceInCurrentDomain'
]
target_column = 'LeaveOrNot'

employee_df = employee_df[[target_column] + feature_columns]



# Display the transformed dataset
employee_df.head()

Education                     int8
JoiningYear                  int64
City                          int8
PaymentTier                  int64
Age                          int64
Gender                        int8
EverBenched                  int64
ExperienceInCurrentDomain    int64
LeaveOrNot                   int64
dtype: object


,LeaveOrNot,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain
0,0,0,2017,0,3,34,1,0,0
1,1,0,2013,2,1,28,0,0,3
2,0,0,2014,1,3,38,0,0,2
3,1,1,2016,0,3,27,1,0,5
4,1,1,2017,2,3,24,1,1,2


In [3]:
from sagemaker.feature_store.feature_group import FeatureGroup
from time import gmtime, strftime, sleep

# Create a Feature Group
feature_group_name = 'employee-feature-group-' + strftime('%Y%m%d%H%M%S', gmtime())
feature_group = FeatureGroup(name=feature_group_name, sagemaker_session=sagemaker_session)

# Define the schema
record_identifier_name = 'EmployeeID'  # Unique identifier for records
event_time_feature_name = 'EventTime'  # Column representing the time of event

# Ensure EventTime is in the correct ISO-8601 format
employee_df[event_time_feature_name] = pd.to_datetime('now').strftime('%Y-%m-%dT%H:%M:%S.%fZ')
employee_df[record_identifier_name] = employee_df.index

# Load features to the Feature Store
feature_group.load_feature_definitions(data_frame=employee_df)

# Enable the Online Store when creating the Feature Group
feature_group.create(
    s3_uri='s3://training-data-ar/features', #change the bucket name here
    record_identifier_name=record_identifier_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True  # Enable the Online Store
)

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:463646775279:feature-group/employee-feature-group-20260508180928',
 'ResponseMetadata': {'RequestId': '3bf9dbc8-2b7b-41c6-b8a3-f9216a672e6e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '3bf9dbc8-2b7b-41c6-b8a3-f9216a672e6e',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '114',
   'date': 'Fri, 08 May 2026 18:09:29 GMT'},
  'RetryAttempts': 0}}

In [5]:
# Check the status of the Feature Group
status = feature_group.describe().get("FeatureGroupStatus")
print(f"Feature Group Status: {status}")

if status == "Created":
    print("Feature Group is Created and ready for use. Proceeding with ingestion...")
    # Ingest data into the Feature Store
    feature_group.ingest(data_frame=employee_df, max_workers=3, wait=True)
    print('Data ingested into Feature Store.')

Feature Group Status: Created
Feature Group is Created and ready for use. Proceeding with ingestion...


Data ingested into Feature Store.


In [6]:
from sagemaker.feature_store.feature_group import FeatureGroup
from sklearn.model_selection import train_test_split 

# Initialize the SageMaker Feature Store runtime client
featurestore_runtime = boto3.client('sagemaker-featurestore-runtime')

# Define the feature group name and features you want to retrieve
feature_names = ['Education', 'JoiningYear', 'City', 'PaymentTier', 'Age', 'Gender', 'EverBenched', 'ExperienceInCurrentDomain', 'LeaveOrNot']

# Retrieve records and convert to DataFrame
records = []
for record_id in employee_df.index.astype(str):
    response = featurestore_runtime.get_record(
        FeatureGroupName=feature_group_name,
        RecordIdentifierValueAsString=str(record_id),
        FeatureNames=feature_names
    )
    # Check if 'Record' is in the response and add to records list
    if 'Record' in response:
        record = {feature['FeatureName']: feature['ValueAsString'] for feature in response['Record']}
        records.append(record)
    else:
        print(f"Record with ID {record_id} not found.")

# Convert the list of records to a DataFrame
retrieved_df = pd.DataFrame(records)

# Check if we have any retrieved records
if not retrieved_df.empty:
    # Split the data into training and test sets
    train_df, test_df = train_test_split(retrieved_df, test_size=0.2, random_state=42)
    print("Training and test data split after retrieval from Feature Store.")
else:
    print("No records retrieved. Please check the feature group and identifiers.")

Training and test data split after retrieval from Feature Store.


## Train the Model Using Local Data with S3 Mode (Default)

In [7]:
#Initialize S3 client
s3 = boto3.client('s3')

# Define your S3 bucket and prefix
bucket = 'training-data-ar'    # change the bucket name here
prefix = 'input-data'

# Save the data locally first
train_file = 'train.csv'
validation_file = 'validation.csv'

train_df.to_csv(train_file, index=False)
test_df.to_csv(validation_file, index=False)

# Upload the data to S3
s3.upload_file(train_file, bucket, f'{prefix}/train/{train_file}')

s3.upload_file(validation_file, bucket, f'{prefix}/validation/{validation_file}')

print(f"Training data uploaded to s3://{bucket}/{prefix}/train/{train_file}")

print(f"Validation data uploaded to s3://{bucket}/{prefix}/validation/{validation_file}")

Training data uploaded to s3://training-data-ar/input-data/train/train.csv
Validation data uploaded to s3://training-data-ar/input-data/validation/validation.csv


In [8]:
import sagemaker
import boto3
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput

# Initialize hyperparameters
hyperparameters = {
        "max_depth": "5",
        "eta": "0.2",
        "gamma": "4",
        "min_child_weight": "6",
        "subsample": "0.7",
        "objective": "reg:squarederror",
        "num_round": "50"}

# Set an output path where the trained model will be saved
bucket = 'training-data-ar'
prefix = 'demo-built-in-algorithm'
output_path = f's3://{bucket}/{prefix}/output'

# Retrieve the XGBoost image URI
region = boto3.Session().region_name  # Automatically get the region
xgboost_container = image_uris.retrieve("xgboost", region, "1.7-1")

# Construct a SageMaker estimator that calls the xgboost-container
estimator = sagemaker.estimator.Estimator(image_uri=xgboost_container, 
                                          hyperparameters=hyperparameters,
                                          role=sagemaker.get_execution_role(),
                                          instance_count=1, 
                                          instance_type='ml.m5.xlarge', 
                                          volume_size=5,  # 5 GB 
                                          output_path=output_path)

# Define the data type and paths to the training and validation datasets
content_type = "csv"
train_input = TrainingInput(f"s3://{bucket}/input-data/train/", content_type=content_type)
validation_input = TrainingInput(f"s3://{bucket}/input-data/validation/", content_type=content_type)

# Execute the XGBoost training job
estimator.fit({'train': train_input, 'validation': validation_input})

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-05-08-18-14-08-546


2026-05-08 18:14:08 Starting - Starting the training job.

.

.


2026-05-08 18:14:28 Starting - Preparing the instances for training.

...


2026-05-08 18:16:07 Training - Training image download completed. Training in progress..

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-08 18:16:15.167 ip-10-0-208-0.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-05-08 18:16:15.228 ip-10-0-208-0.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-05-08:18:16:15:INFO] Imported framework sagemaker_xgboost_container.training
[2026-05-08:18:16:15:INFO] Failed to parse hyperparameter objective value reg:squarederror to Json.
Returning the value itself
[2026-05-08:18:16:15:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-08:18:16:15:INFO] Running XGBoost Sagemaker in algorithm mode
[2026-05-08:18:16:15:INFO] Determined 0 GPU(s) available on the instance.
[202


2026-05-08 18:16:36 Uploading - Uploading generated training model
2026-05-08 18:16:36 Completed - Training job completed


Training seconds: 105
Billable seconds: 105


# Hyperparameter Tuning Script

In [9]:
# Import SageMaker, AWS SDK, and tuning utilities
import sagemaker
import boto3
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter


# ---------------------------------------------------
# Initialize SageMaker session and execution role
# ---------------------------------------------------

# Create SageMaker session
sagemaker_session = sagemaker.Session()

# Get IAM execution role for SageMaker
role = sagemaker.get_execution_role()


# ---------------------------------------------------
# Define fixed/base hyperparameters
# ---------------------------------------------------

# These parameters remain constant during tuning
base_hyperparameters = {
    "objective": "reg:squarederror",  # Regression objective
    "num_round": 50                   # Number of boosting rounds
}


# ---------------------------------------------------
# Retrieve XGBoost container image
# ---------------------------------------------------

# Automatically detect AWS region
region = boto3.Session().region_name

# Get the built-in SageMaker XGBoost image URI
xgboost_container = image_uris.retrieve("xgboost", region, "1.7-1")


# ---------------------------------------------------
# Configure the SageMaker Estimator
# ---------------------------------------------------

# Estimator defines the training environment
estimator = sagemaker.estimator.Estimator(
    image_uri=xgboost_container,              # XGBoost container
    hyperparameters=base_hyperparameters,     # Fixed hyperparameters
    role=role,                                # IAM role
    instance_count=1,                         # Number of instances
    instance_type='ml.m5.xlarge',             # Compute instance type
    volume_size=5,                            # Storage size in GB
    output_path=f's3://{bucket}/{prefix}/output',  # S3 output location
    sagemaker_session=sagemaker_session
)


# ---------------------------------------------------
# Define hyperparameter search ranges
# ---------------------------------------------------

# SageMaker will test different combinations
hyperparameter_ranges = {

    # Maximum depth of decision trees
    'max_depth': IntegerParameter(3, 10),

    # Learning rate / step size
    'eta': ContinuousParameter(0.1, 0.5),

    # Minimum loss reduction required for split
    'gamma': ContinuousParameter(0, 5),

    # Minimum sum of instance weight in child
    'min_child_weight': IntegerParameter(1, 10),

    # Fraction of training data used per tree
    'subsample': ContinuousParameter(0.5, 1.0)
}


# ---------------------------------------------------
# Define evaluation metric
# ---------------------------------------------------

# Extract RMSE metric from training logs
metric_definitions = [{
    'Name': 'validation:rmse',
    'Regex': '.*\\[validation-rmse\\] ([0-9\\.]+)'
}]

# Metric to optimize during tuning
objective_metric_name = 'validation:rmse'


# ---------------------------------------------------
# Create the Hyperparameter Tuner
# ---------------------------------------------------

# Configure tuning strategy
tuner = HyperparameterTuner(
    estimator=estimator,                      # Base estimator
    objective_metric_name=objective_metric_name,  # Metric to optimize
    hyperparameter_ranges=hyperparameter_ranges,  # Search space
    metric_definitions=metric_definitions,
    objective_type='Minimize',                # Lower RMSE is better
    max_jobs=20,                              # Total training jobs
    max_parallel_jobs=3                       # Parallel jobs running together
)


# ---------------------------------------------------
# Define training and validation datasets
# ---------------------------------------------------

# Training dataset stored in S3
train_input = TrainingInput(
    f"s3://{bucket}/input-data/train/",
    content_type="csv"
)

# Validation dataset stored in S3
validation_input = TrainingInput(
    f"s3://{bucket}/input-data/validation/",
    content_type="csv"
)


# ---------------------------------------------------
# Start Hyperparameter Tuning Job
# ---------------------------------------------------

# Launch tuning process
tuner.fit(
    {
        'train': train_input,
        'validation': validation_input
    },

    # Unique tuning job name
    job_name='xgb-tuning-job'
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker:Creating hyperparameter tuning job with name: xgb-tuning-job


.

.

.

...

.

.

.

.

!

In [10]:
# Import AWS SDK for Python
import boto3


# ---------------------------------------------------
# Initialize SageMaker client
# ---------------------------------------------------

# Create a SageMaker client using Boto3
client = boto3.client('sagemaker')


# ---------------------------------------------------
# Specify Hyperparameter Tuning Job Name
# ---------------------------------------------------

# Replace with your actual tuning job name
tuning_job_name = 'xgb-tuning-job'   # Change your training job name here


# ---------------------------------------------------
# Retrieve tuning job details
# ---------------------------------------------------

# Get information about the hyperparameter tuning job
response = client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)


# ---------------------------------------------------
# Retrieve the best training job
# ---------------------------------------------------

# SageMaker automatically identifies the best-performing job
best_training_job = response['BestTrainingJob']

# Print summary of the best training job
print("Best Training Job:", best_training_job)


# ---------------------------------------------------
# Get detailed information about the best model
# ---------------------------------------------------

# Extract the best training job name
best_training_job_name = best_training_job['TrainingJobName']

# Retrieve detailed training job information
best_training_job_response = client.describe_training_job(
    TrainingJobName=best_training_job_name
)

# Print complete training job details
print("Best Training Job Details:", best_training_job_response)


# ---------------------------------------------------
# Retrieve evaluation metrics
# ---------------------------------------------------

# Get final metrics generated during training
metrics = best_training_job_response.get('FinalMetricDataList', [])


# ---------------------------------------------------
# Display metric results
# ---------------------------------------------------

# Print all evaluation metrics and their values
for metric in metrics:
    print(
        f"Metric Name: {metric['MetricName']}, "
        f"Value: {metric['Value']}"
    )

Best Training Job: {'TrainingJobName': 'xgb-tuning-job-036-64c3f84e', 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:463646775279:training-job/xgb-tuning-job-036-64c3f84e', 'CreationTime': datetime.datetime(2026, 5, 8, 18, 42, 43, tzinfo=tzlocal()), 'TrainingStartTime': datetime.datetime(2026, 5, 8, 18, 42, 50, tzinfo=tzlocal()), 'TrainingEndTime': datetime.datetime(2026, 5, 8, 18, 43, 24, tzinfo=tzlocal()), 'TrainingJobStatus': 'Completed', 'TunedHyperParameters': {'eta': '0.46441314308991577', 'gamma': '1.212371863280548', 'max_depth': '9', 'min_child_weight': '10', 'subsample': '0.9695241877646337'}, 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:rmse', 'Value': 0.3135699927806854}, 'ObjectiveStatus': 'Succeeded'}
Best Training Job Details: {'TrainingJobName': 'xgb-tuning-job-036-64c3f84e', 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:463646775279:training-job/xgb-tuning-job-036-64c3f84e', 'TuningJobArn': 'arn:aws:sagemaker:us-east-1:463646775279:hyper-p

In [11]:

import boto3

# ---------------------------------------------------
# Initialize SageMaker client
# ---------------------------------------------------
# Create a SageMaker client
client = boto3.client('sagemaker')

# ---------------------------------------------------
# Specify the Hyperparameter Tuning Job
# ---------------------------------------------------

# Name of the completed tuning job
tuning_job_name = 'xgb-tuning-job'  # Replace if needed

# ---------------------------------------------------
# Retrieve tuning job details
# ---------------------------------------------------

# Get information about the tuning job
response = client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)

# ---------------------------------------------------
# Retrieve the best training job
# ---------------------------------------------------

# SageMaker selects the best-performing training job
best_training_job = response['BestTrainingJob']

# Extract the training job name
best_training_job_name = best_training_job['TrainingJobName']

# ---------------------------------------------------
# Retrieve details of the best training job
# ---------------------------------------------------

# Get detailed information for the best model
best_training_job_response = client.describe_training_job(
    TrainingJobName=best_training_job_name
)

# ---------------------------------------------------
# Extract best hyperparameters
# ---------------------------------------------------

# Retrieve optimized hyperparameters from the best job
best_hyperparameters = best_training_job_response['HyperParameters']

# ---------------------------------------------------
# Display best hyperparameters
# ---------------------------------------------------

# Print the optimal hyperparameter values
print("Best Hyperparameters:", best_hyperparameters)

Best Hyperparameters: {'_tuning_objective_metric': 'validation:rmse', 'eta': '0.46441314308991577', 'gamma': '1.212371863280548', 'max_depth': '9', 'min_child_weight': '10', 'num_round': '50', 'objective': 'reg:squarederror', 'subsample': '0.9695241877646337'}
